In [ ]:
import os,sys,json,hashlib,zipfile,io,subprocess,shutil,time
from pathlib import Path
SESSION_STARTED=time.monotonic()
# Must be supplied by the trusted dispatch operator after a separate owner grant.
AUTHORIZATION={'GPU_authorized': True, 'evidence_source': 'explicit_human_owner_message', 'owner_authorization_reference': 'Current owner message: 我明确授权GPU推理. This is the separate explicit grant for the exact six-session/two-batch experiment specified in the accompanying pasted task.', 'human_message_reference': 'Current owner message: 我明确授权GPU推理. This is the separate explicit grant for the exact six-session/two-batch experiment specified in the accompanying pasted task.', 'experiment_id': 'MEER-V01-MATCHED-PILOT', 'manifest_sha256': '69616f1ceae3613118699793ee8d61e723bb3a32c09302a68891fb714453c059', 'payload_sha256': '38170bc0d6684ea09097d106cb0a4fbcf5fbc67a0b7c061f139ae99bccc558b5', 'batch': 1, 'order': [{'run': 1, 'instance_id': 'fastapi_14297', 'arm': 'A'}, {'run': 2, 'instance_id': 'fastapi_14297', 'arm': 'B'}, {'run': 3, 'instance_id': 'fastapi_14297', 'arm': 'C'}], 'arms': {'A': {'ZIP_sha256': '07263740c5ee22bf6611d7fa31dafbc84d7fb50288d22355b9201cd6954a6840', 'members': {'agent.yaml': '7ebd41cd35a20ae46aaa8dba1db23bb528250e285a4a4cdc9dc1ea67eceaaf15', 'eval_config.yaml': '4336195f5a2c63071dd2ee2448d13f6630c4a9f81ba4afc399dedb8312785861', 'prompts/system.md': 'b1dceeca98db3353d1f0055c7ddd26be9fdcde61a57f0dc22f43f0e9aed0a7e0'}}, 'B': {'ZIP_sha256': 'd962044f264d8f36aea6263abf4dcdb118d5755e82a91cb9b2bd0bc1f6f7c9cd', 'members': {'agent.yaml': '5e5fbafca6ea09c9cb38137aff54ddcd5fc1e2678afd166b95c0c88294a62b04', 'eval_config.yaml': '4336195f5a2c63071dd2ee2448d13f6630c4a9f81ba4afc399dedb8312785861', 'prompts/system.md': 'b1dceeca98db3353d1f0055c7ddd26be9fdcde61a57f0dc22f43f0e9aed0a7e0', 'skills/meer-evidence/SKILL.md': 'bdb2719c8f1768a5f8f90626ed7f28d32133a06bbb59afbef46686047b0af8dc', 'skills/meer-evidence/scripts/verify.py': 'dae959c8068e6d320c7fc5c37f719ccd928f56dd19f4335af0d33193ea3f499c'}}, 'C': {'ZIP_sha256': '8dd48fee0c96424a40ed47512f0d788d9b13518e7d73395bd89d2fb07204eb83', 'members': {'agent.yaml': '5e5fbafca6ea09c9cb38137aff54ddcd5fc1e2678afd166b95c0c88294a62b04', 'eval_config.yaml': '4336195f5a2c63071dd2ee2448d13f6630c4a9f81ba4afc399dedb8312785861', 'prompts/system.md': 'b1dceeca98db3353d1f0055c7ddd26be9fdcde61a57f0dc22f43f0e9aed0a7e0', 'skills/meer-evidence/SKILL.md': 'bdb2719c8f1768a5f8f90626ed7f28d32133a06bbb59afbef46686047b0af8dc', 'skills/meer-evidence/scripts/verify.py': '8e7802ec2033b7e438300a32c0c3181eadae81814f8ded650452ba25addbef1d'}}}, 'environment_lock_sha256': '452c0d9afdf7c21edcd43a6a67e69cca6c67aba640eb526f50aa9835169dc7d9', 'model_files': {'README.md': 'b68c4f66b6bbe75f72f54a5aed65ed08a7faf6ca056bdf11beefedf8591adad3', 'chat_template.jinja': '94899c0f917d93f6fe81c95744d1e8ddab2d21d39228d2e4aec1fb2a25bff413', 'config.json': 'b100d85e571c25b688e82919d819253063a1defb0e6cee3b9b1c1fbad99bd0c2', 'generation_config.json': '70110aa886fa7ee25ce47346d102dd29ca5f967500490b4362fc72068e1db70a', 'model.safetensors': '1b9b1d622a93f02c0d33f98e502f233b5d707443af6ddc464ed0bf5498506c20', 'processor_config.json': '32bdf45d2ad4cc29a0822ddd157a182de76644f0419a6228d151495256e9813c', 'tokenizer.json': 'cc8d3a0ce36466ccc1278bf987df5f71db1719b9ca6b4118264f45cb627bfe0f', 'tokenizer_config.json': 'a284d1243b62be31faa9c13e1c28cece940c4abaa7bd9ad87b94f61b40687200'}, 'max_SaveKernel_dispatches': 1, 'max_Agent_runs': 3, 'GPU_count': 4, 'GPU_type': 'NVIDIA L4', 'TP': 4, 'Agent_seconds': 300, 'platform_window_seconds': 10000, 'quota_at_dispatch_report_seconds': '108000', 'authorization_record_sha256': '237806a6cd606c7b51abe2a6e2243c0522d77a07d5229ba6da2829c6dae45e84'}
assert AUTHORIZATION and AUTHORIZATION.get('GPU_authorized') is True, 'Gate F: GPU not authorized'
RUN_DIR=Path('/kaggle/working/meer-v01-gpu-batch1');RUN_DIR.mkdir(parents=True,exist_ok=False)
SOURCE=RUN_DIR/'source'
payload=__import__('base64').b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='38170bc0d6684ea09097d106cb0a4fbcf5fbc67a0b7c061f139ae99bccc558b5'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None and all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(SOURCE)
sys.path.insert(0,str(SOURCE))
REG=json.loads((SOURCE/'gpu_registration.json').read_bytes())
for n,h in REG['files'].items():assert hashlib.sha256((SOURCE/n).read_bytes()).hexdigest()==h,n
identity=dict(payload_sha256='38170bc0d6684ea09097d106cb0a4fbcf5fbc67a0b7c061f139ae99bccc558b5',production_entry=REG['production_entry'])
(RUN_DIR/'PAYLOAD_IDENTITY.json').write_text(json.dumps(identity))
from meer.v01.gpu01.session import authorize
authorize(AUTHORIZATION,json.loads((SOURCE/'meer/v01/gpu01/EXECUTION_MANIFEST.json').read_bytes()),1,identity)
WHEELHOUSE_DIR=next(Path('/kaggle/input').rglob('swegemma-0.2.11-py3-none-any.whl')).parent
assert {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in WHEELHOUSE_DIR.glob('*.whl') if 'cutlass' not in p.name.lower()}==REG['wheel_hashes']


In [ ]:
import glob
os.environ.update(TRANSFORMERS_NO_TF='1',TRANSFORMERS_NO_FLAX='1',TRANSFORMERS_OFFLINE='1',HF_HUB_OFFLINE='1',LITELLM_LOCAL_MODEL_COST_MAP='True',LITELLM_TELEMETRY='False',OTEL_SDK_DISABLED='true',PYTHONIOENCODING='utf-8')
for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth','/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for path in glob.glob(pattern):Path(path).unlink(missing_ok=True)
links=RUN_DIR/'wheelhouse_links';links.mkdir()
try:
    for p in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in p.name.lower():continue
        name=p.name.replace('cu128','+cu128') if 'cu128' in p.name and '+' not in p.name else p.name
        (links/name).symlink_to(p)
    with (RUN_DIR/'installation.log').open('x') as log:
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall']+sorted(str(p) for p in links.glob('*.whl')),stdout=log,stderr=subprocess.STDOUT,check=True,timeout=1800)
finally:shutil.rmtree(links)
import asyncio
from harness_v2.evidence import bundle
from meer.v01.gpu01.session import run
try:
    asyncio.run(run(RUN_DIR/'execution',SESSION_STARTED,batch=1,authorization=AUTHORIZATION,quota_at_dispatch=AUTHORIZATION['quota_at_dispatch_report_seconds']))
finally:
    print('MEER_GPU_EVIDENCE',bundle(RUN_DIR),flush=True)
